# Semana 5 — SQL sobre FHIR: vistas tabulares estándar (SQL on FHIR) y cuándo usar API, SQL u OMOP

**Pregunta de la semana:** tienes datos clínicos en FHIR. ¿Cómo los consultas con SQL sin perder lo que FHIR sabe (referencias, tipos de elección, códigos), y cuándo conviene la API REST, cuándo SQL y cuándo un modelo común como OMOP?

**Qué es SQL on FHIR:** una especificación de HL7 que define el recurso `ViewDefinition`: una descripción portable de cómo convertir **un tipo de recurso** FHIR en una tabla plana, con expresiones FHIRPath para columnas, filtros y desanidado. Las uniones (*joins*), el orden y las agregaciones **no** se hacen en la vista: se hacen después, en el motor SQL. La versión que usamos como referencia es la 3.0.0-ballot (*Trial use*), publicada por HL7 International / FHIR Infrastructure.

**Flujo de trabajo de esta semana (las tres capas de la especificación):**

1. **Capa de datos:** conviertes los Bundles de Synthea de la semana 4 en archivos NDJSON por tipo de recurso, el formato que produce una exportación masiva FHIR (`$export`).
2. **Capa de vistas:** escribes `ViewDefinition`s y las ejecutas con un *view runner* pequeño que tú mismo construyes y compruebas contra el conjunto de pruebas oficial.
3. **Capa analítica:** cargas las tablas en DuckDB, las unes con SQL, las concilias con las tablas de la semana 4 y comparas con SQL directo sobre JSON y con la API REST.

**Al terminar tendrás:** `data/fhir_ndjson/*.ndjson` con su manifiesto, `views/*.json` (las vistas, que sí van al repo), `data/sof/*.parquet` y `data/reports/sof_conformance.csv`.

**Requisitos:** semanas 3 y 4 (`data/raw/synthea_sample_data_fhir_r4_nov2021.zip` y `data/silver_fhir/`).

### Celda de configuración del entorno (se repite al inicio de cada semana)

**Explicación línea por línea**

- `import os`, `import sys`, `from pathlib import Path`: cargan tres módulos de la librería estándar. `os` lee variables de entorno, `sys` informa la versión de Python y `Path` representa rutas de archivos como objetos. **Por qué así:** `Path` evita concatenar rutas con texto (`"a" + "/" + "b"`), que falla entre sistemas operativos. **En la práctica real** todo código de datos moderno usa `pathlib`.
- `try: import google.colab` … `except ImportError`: intenta importar un módulo que solo existe en Colab. Si la importación funciona, `IN_COLAB = True`; si lanza `ImportError`, estamos en otra máquina. **Por qué importa:** el mismo notebook corre en Colab y en tu laptop sin editarlo. **Cuándo se usa:** siempre que un código deba ser portable entre entornos.
- `from google.colab import drive` y `drive.mount("/content/drive")`: conectan tu Google Drive al sistema de archivos de Colab. **Por qué:** el disco de Colab se borra al cerrar la sesión; en Drive los datos de la semana 1 siguen ahí el lunes siguiente. La primera vez Colab te pedirá autorizar el acceso.
- `PROJECT_DIR = Path("/content/drive/MyDrive/health-data-etl-lab")`: carpeta raíz del proyecto dentro de tu Drive.
- `Path(os.environ.get("HEALTHETL_PROJECT_DIR", ".")).resolve()`: fuera de Colab usa la variable de entorno `HEALTHETL_PROJECT_DIR` si existe; si no, la carpeta actual (`"."`). `.resolve()` la convierte en ruta absoluta. **En la práctica real** la configuración se pasa por variables de entorno, nunca escrita a mano dentro del código (principio *12-factor app*).
- `DATA_DIR = PROJECT_DIR / "data"`: el operador `/` de `pathlib` une rutas de forma segura.
- `DATA_DIR.mkdir(parents=True, exist_ok=True)`: crea la carpeta y sus padres; `exist_ok=True` evita un error si ya existe. Esto hace la celda **idempotente**: puedes ejecutarla 1 o 10 veces con el mismo resultado, una propiedad central de todo pipeline ETL.
- Los tres `print`: confirman versión de Python, entorno y ruta. Registrar el entorno al inicio es un hábito de reproducibilidad.

In [ ]:
import os
import sys
from pathlib import Path

try:
    import google.colab  # este módulo solo existe dentro de Google Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/health-data-etl-lab")
else:
    PROJECT_DIR = Path(os.environ.get("HEALTHETL_PROJECT_DIR", ".")).resolve()

DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Python:", sys.version.split()[0])
print("¿Colab?:", IN_COLAB)
print("Carpeta de datos:", DATA_DIR)

### Paso 1 — Instalar el motor FHIRPath y preparar carpetas

**Explicación línea por línea**

- `%pip install -q "fhirpathpy==2.2.4"`: instala `fhirpathpy` (licencia MIT, de beda.software), una implementación de FHIRPath en Python. **Versión fija:** si mañana sale otra versión con un cambio de comportamiento, tu notebook seguirá dando los mismos resultados. En la práctica real se fija la versión de toda dependencia que afecta a los datos.
- `from fhirpathpy import compile as fp_compile`: `compile` traduce una expresión FHIRPath una sola vez a una función reutilizable. Se renombra a `fp_compile` para no ocultar la función `compile` de Python.
- `from fhirpathpy.models import models`: los "modelos" describen los tipos de cada versión de FHIR. Sin el modelo R4, FHIRPath no sabe que `value.ofType(Quantity)` corresponde al elemento JSON `valueQuantity` (los **tipos de elección** `value[x]`).
- `NDJSON_DIR`, `SOF_DIR`, `REPORTS_DIR`, `EXT_DIR`: salidas de cada capa. `VIEWS_DIR = PROJECT_DIR / "views"` queda **fuera** de `data/`: las vistas son metadatos sin datos de pacientes y se versionan en git.

In [ ]:
%pip install -q "fhirpathpy==2.2.4"

In [ ]:
import hashlib
import json
import subprocess
from decimal import Decimal
import time
import zipfile

import duckdb
import pandas as pd
from fhirpathpy import compile as fp_compile
from fhirpathpy.engine.nodes import ResourceNode
from fhirpathpy.models import models

FHIR_ZIP = DATA_DIR / "raw" / "synthea_sample_data_fhir_r4_nov2021.zip"
NDJSON_DIR = DATA_DIR / "fhir_ndjson"
SOF_DIR = DATA_DIR / "sof"
REPORTS_DIR = DATA_DIR / "reports"
EXT_DIR = DATA_DIR / "external"
VIEWS_DIR = PROJECT_DIR / "views"
for d in (NDJSON_DIR, SOF_DIR, REPORTS_DIR, EXT_DIR, VIEWS_DIR):
    d.mkdir(parents=True, exist_ok=True)
assert FHIR_ZIP.exists(), "Ejecuta primero la semana 4: falta el ZIP FHIR de Synthea"
print("ZIP FHIR:", FHIR_ZIP.name)

## Parte A — Capa de datos: de Bundles de transacción a NDJSON

### Paso 2 — Simular la salida de una exportación masiva (`$export`)

En un sistema real no descargarías un ZIP: pedirías al servidor una **exportación masiva**. La operación `$export` es asíncrona y, al terminar, devuelve un manifiesto con enlaces a archivos NDJSON, uno o más por tipo de recurso. NDJSON es un objeto JSON por línea: se puede leer por streaming sin cargar el archivo completo en memoria.

Los Bundles de Synthea son de tipo `transaction`: los recursos se referencian entre sí con identificadores temporales `urn:uuid:...`. Cuando un servidor procesa una transacción y asigna identificadores, la especificación le exige actualizar las referencias dentro del Bundle. Por eso aquí reemplazas cada `urn:uuid:...` por la referencia relativa `Tipo/id`, que es lo que contendría un `$export` real.

**Explicación línea por línea**

- `TYPES = ["Patient", "Encounter", "Condition", "Observation"]`: los tipos que usarás. Exportar solo lo necesario es lo que hace el parámetro `_type` de `$export`.
- `def resolve_refs(node, urn_map):` recorre el recurso de forma **recursiva** (diccionarios dentro de listas dentro de diccionarios). Si encuentra una clave `"reference"` cuyo valor está en `urn_map`, lo sustituye por `Tipo/id`. Recursiva porque las referencias pueden estar a cualquier profundidad (`subject`, `encounter`, `context.related`...).
- `urn_map = {e["fullUrl"]: f'{e["resource"]["resourceType"]}/{e["resource"]["id"]}' ...}`: el mapa de cada Bundle, de `urn:uuid:abc` a `Patient/abc`.
- `writers = {t: open(..., "w", encoding="utf-8") for t in TYPES}`: un archivo abierto por tipo; se escribe a medida que se lee (streaming).
- `json.dumps(res, separators=(",", ":"))`: una línea compacta por recurso, sin espacios.
- `unresolved`: cuenta referencias que siguieron empezando por `urn:` después del reemplazo. Debe ser 0; si no, esas filas perderían su relación con el paciente.
- `manifest = {...}`: imita la estructura del manifiesto de `$export` (`transactionTime`, `request`, `output` con `type`, `url` y `count`) y añade la huella SHA-256 de cada archivo para el linaje. Lleva `"simulated": True` para que nadie lo confunda con una exportación real.
- `if manifest_path.exists(): ...`: **idempotencia**. Si ya hiciste la exportación con este mismo ZIP, no se repite.

In [ ]:
TYPES = ["Patient", "Encounter", "Condition", "Observation"]


def resolve_refs(node, urn_map):
    if isinstance(node, dict):
        out = {}
        for k, v in node.items():
            if k == "reference" and isinstance(v, str) and v in urn_map:
                out[k] = urn_map[v]
            else:
                out[k] = resolve_refs(v, urn_map)
        return out
    if isinstance(node, list):
        return [resolve_refs(x, urn_map) for x in node]
    return node


def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


manifest_path = NDJSON_DIR / "export_manifest.json"
zip_sha = sha256_of(FHIR_ZIP)
if manifest_path.exists() and json.loads(manifest_path.read_text())["source_zip_sha256"] == zip_sha:
    manifest = json.loads(manifest_path.read_text())
    print("Exportación ya realizada con este ZIP: se reutiliza")
else:
    counts = {t: 0 for t in TYPES}
    unresolved = 0
    t0 = time.time()
    with zipfile.ZipFile(FHIR_ZIP) as zf:
        names = sorted(n for n in zf.namelist() if n.endswith(".json"))
        writers = {t: open(NDJSON_DIR / f"{t}.ndjson", "w", encoding="utf-8") for t in TYPES}
        try:
            for name in names:
                bundle = json.loads(zf.read(name))
                entries = bundle.get("entry", [])
                urn_map = {e["fullUrl"]: f'{e["resource"]["resourceType"]}/{e["resource"]["id"]}'
                           for e in entries if e.get("fullUrl", "").startswith("urn:uuid:")}
                for e in entries:
                    res = e["resource"]
                    if res["resourceType"] not in writers:
                        continue
                    res = resolve_refs(res, urn_map)
                    line = json.dumps(res, separators=(",", ":"))
                    unresolved += line.count('"reference":"urn:')
                    writers[res["resourceType"]].write(line + "\n")
                    counts[res["resourceType"]] += 1
        finally:
            for w in writers.values():
                w.close()
    assert unresolved == 0, f"{unresolved} referencias sin resolver"
    manifest = {
        "simulated": True,
        "transactionTime": pd.Timestamp.now(tz="UTC").isoformat(),
        "request": f"[base]/$export?_type={','.join(TYPES)}",
        "source_zip_sha256": zip_sha,
        "output": [{"type": t, "url": f"{t}.ndjson", "count": counts[t],
                    "sha256": sha256_of(NDJSON_DIR / f"{t}.ndjson")} for t in TYPES],
        "error": [],
    }
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print(f"Exportación simulada en {time.time() - t0:.0f} s de {len(names)} Bundles")
{o["type"]: o["count"] for o in manifest["output"]}

### Paso 3 — Mirar una línea: referencias ya resueltas

**Explicación línea por línea**

- `with open(...) as fh: first = json.loads(fh.readline())`: lee **solo la primera línea** de `Condition.ndjson`. Con NDJSON no hace falta abrir el archivo completo para inspeccionarlo.
- `first["subject"]`, `first["encounter"]`: ahora dicen `Patient/...` y `Encounter/...`, no `urn:uuid:...`.
- `first["code"]`: un `CodeableConcept` con su lista `coding` (sistema + código + texto). Recuerda la estructura: la usarás en las vistas.

In [ ]:
with open(NDJSON_DIR / "Condition.ndjson", encoding="utf-8") as fh:
    first = json.loads(fh.readline())
{k: first[k] for k in ["resourceType", "id", "subject", "encounter", "code", "clinicalStatus"] if k in first}

## Parte B — Capa de vistas: un *view runner* propio, comprobado contra las pruebas oficiales

### Paso 4 — El *view runner*

La especificación describe un modelo funcional pequeño: para cada recurso del tipo de la vista que cumpla los filtros `where`, se evalúa cada `select`. Un `select` produce filas con sus `column`s; `forEach` genera una fila por elemento de una colección y descarta el recurso si la colección está vacía; `forEachOrNull` conserva una fila con nulos; varios `select` del mismo nivel se combinan como producto cartesiano, y `unionAll` concatena filas de ramas con las mismas columnas. Además, todo *view runner* debe implementar dos funciones que no son de FHIRPath: `getResourceKey()` (la clave de la fila) y `getReferenceKey([tipo])` (la clave del recurso referenciado), para que las vistas se puedan unir después.

**Explicación línea por línea**

- `class ViewError(Exception)`: un error propio para fallos de la vista (no de Python).
- `def _resource_key(inputs):` devuelve el `id` del recurso. La especificación deja la clave a criterio de cada implementación; el `id` es la opción habitual.
- `def _reference_key(inputs, type_spec=None):` toma `Reference.reference` en forma relativa (`Patient/123`), la parte en `["Patient", "123"]` y devuelve `123`. Si se pidió un tipo (`getReferenceKey(Patient)`) y la referencia es de otro, devuelve vacío. `getattr(type_spec, "name", None)`: `fhirpathpy` entrega el tipo como un objeto con atributo `name`.
- `UDF = {...}`: la tabla de funciones de usuario de `fhirpathpy`. `"arity": {0: [], 1: ["TypeSpecifier"]}` declara que `getReferenceKey` acepta cero o un argumento de tipo.
- `_cache = {}` y `def fp(node, path, consts, raw=False):` compilan cada expresión **una sola vez** (la clave del caché es la expresión más el modo). Medido en este laboratorio: evaluar sin compilar costó unos 2 ms por observación y compilado unos 0,14 ms. Con 200.000 observaciones, esa diferencia separa minutos de horas.
- `if path.strip() == "$this": return [node]`: `$this` es "el elemento actual". Se resuelve directamente porque `fhirpathpy` solo lo define dentro de funciones como `where()`.
- `raw=True` y `"returnRawData"`: en `forEach` los elementos se devuelven como nodos que **recuerdan su ruta** dentro del recurso (`Observation.component`). Un diccionario suelto no sabe que es un componente de `Observation`, y entonces `value.ofType(Quantity)` no puede saber que debe leer `valueQuantity` y devuelve vacío. **Este error se detectó al conciliar con la semana 4** (Paso 8): los componentes salían sin valor. Es exactamente el tipo de fallo silencioso que la conciliación existe para atrapar.
- `def _unwrap(v):` convierte el nodo en datos simples (`convert_data()`) y los `Decimal` en `float`. FHIRPath trabaja con decimales exactos y `fhirpathpy` los devuelve como `Decimal`; para una tabla analítica se usa `float`, el mismo tipo que en la semana 4.
- `def _columns(sel):` lista todas las columnas que produce un `select`, incluidas las de sus `select` anidados. Hace falta para fabricar la fila de nulos de `forEachOrNull` y para comprobar que las ramas de `unionAll` coinciden.
- `def _select(sel, node, consts):`
  - `items = fp(node, sel["forEach"], consts)`: los elementos sobre los que se itera; sin `forEach`, el propio nodo;
  - `if col.get("collection")`: una columna marcada como colección guarda una lista; si no lo está y devuelve más de un valor, **es un error** (la especificación lo exige: mejor fallar que guardar el primer valor al azar);
  - `partial = [{**a, **b} for a in partial for b in _select(sub, item, consts)]`: producto cartesiano con los `select` anidados;
  - `if any(c != branch_cols[0] ...)`: las ramas de `unionAll` deben tener las mismas columnas y en el mismo orden.
- `def run_view(view, resources):` filtra por `resourceType` y por `where`: cada condición debe ser booleana (si no, es un error de la vista) y todas deben dar exactamente `[True]`. Después combina los `select` de primer nivel.
- `constant`: cada constante se expone en FHIRPath como `%nombre`.

In [ ]:
R4 = models["r4"]


class ViewError(Exception):
    pass


def _resource_key(inputs):
    return [r["id"] for r in inputs if isinstance(r, dict) and "id" in r]


def _reference_key(inputs, type_spec=None):
    type_name = getattr(type_spec, "name", None)
    out = []
    for ref in inputs:
        literal = ref.get("reference", "") if isinstance(ref, dict) else ""
        parts = literal.split("/")
        if len(parts) >= 2 and (type_name is None or parts[-2] == type_name):
            out.append(parts[-1])
    return out


UDF = {
    "getResourceKey": {"fn": _resource_key, "arity": {0: []}},
    "getReferenceKey": {"fn": _reference_key, "arity": {0: [], 1: ["TypeSpecifier"]}},
}
_cache = {}


def _unwrap(v):
    if isinstance(v, ResourceNode):
        v = v.convert_data()
    return float(v) if isinstance(v, Decimal) else v


def fp(node, path, consts, raw=False):
    if path.strip() == "$this":
        return [node] if raw else [_unwrap(node)]
    key = (path, raw)
    if key not in _cache:
        _cache[key] = fp_compile(path, model=R4, options={"userInvocationTable": UDF, "returnRawData": raw})
    out = _cache[key](node, consts)
    return out if raw else [_unwrap(v) for v in out]


def _columns(sel):
    names = [c["name"] for c in sel.get("column", [])]
    for sub in sel.get("select", []):
        names += _columns(sub)
    if sel.get("unionAll"):
        names += _columns(sel["unionAll"][0])
    return names


def _select(sel, node, consts):
    if "forEach" in sel:
        items = fp(node, sel["forEach"], consts, raw=True)
    elif "forEachOrNull" in sel:
        items = fp(node, sel["forEachOrNull"], consts, raw=True) or [None]
    else:
        items = [node]
    rows = []
    for item in items:
        if item is None:
            rows.append({name: None for name in _columns(sel)})
            continue
        row = {}
        for col in sel.get("column", []):
            values = fp(item, col["path"], consts)
            if col.get("collection"):
                row[col["name"]] = values
            elif len(values) > 1:
                raise ViewError(f"La columna {col['name']} devolvió {len(values)} valores")
            else:
                row[col["name"]] = values[0] if values else None
        partial = [row]
        for sub in sel.get("select", []):
            partial = [{**a, **b} for a in partial for b in _select(sub, item, consts)]
        if sel.get("unionAll"):
            branch_cols = [_columns(b) for b in sel["unionAll"]]
            if any(c != branch_cols[0] for c in branch_cols):
                raise ViewError("Las ramas de unionAll deben tener las mismas columnas en el mismo orden")
            union = [r for branch in sel["unionAll"] for r in _select(branch, item, consts)]
            partial = [{**a, **b} for a in partial for b in union]
        rows.extend(partial)
    return rows


def run_view(view, resources):
    consts = {c["name"]: next(v for k, v in c.items() if k.startswith("value")) for c in view.get("constant", [])}
    out = []
    for res in resources:
        if res.get("resourceType") != view["resource"]:
            continue
        keep = True
        for w in view.get("where", []):
            vals = fp(res, w["path"], consts)
            if any(not isinstance(x, bool) for x in vals):
                raise ViewError(f"where debe ser booleano: {w['path']}")
            keep = keep and vals == [True]
        if not keep:
            continue
        rows = [{}]
        for sel in view.get("select", []):
            rows = [{**a, **b} for a in rows for b in _select(sel, res, consts)]
        out.extend(rows)
    return out


print("View runner listo")

### Paso 5 — Conformidad: pasar las pruebas oficiales

Un *view runner* propio no vale nada si no demuestra que interpreta la especificación igual que los demás. El grupo de trabajo publica un conjunto de pruebas compartido (archivos JSON con recursos de entrada, una vista y las filas esperadas) en el repositorio de la implementación de referencia en JavaScript, `FHIR/sql-on-fhir.js` (licencia MIT).

**Explicación línea por línea**

- `git clone --depth 1`: descarga solo la última versión. `git -C ... rev-parse HEAD` guarda el *commit* exacto en el reporte: las pruebas cambian con el tiempo y sin el commit no podrías repetir el resultado.
- `IMPLEMENTED = [...]`: los archivos de prueba que cubren lo que tu runner implementa. **Fuera de alcance, declarado:** `repeat` (recorrido recursivo), `row_index` (`%rowIndex`) y `fn_boundary` (`lowBoundary()`/`highBoundary()`). Declarar el alcance es lo correcto: un runner que dice cumplir "todo" sin pruebas no es verificable.
- `def norm(rows):` convierte cada fila en JSON con claves ordenadas y ordena la lista: se comparan las filas **como conjunto**, sin depender del orden de salida (la vista no define orden).
- `rows = run_view(...)` dentro de `try`: se ejecuta la vista y luego se compara con `t["expect"]`.
- `t.get("expectError")`: algunas pruebas exigen que el runner **falle** (por ejemplo, ramas de `unionAll` con columnas distintas o un `where` que no es booleano). Si el runner lanza una excepción en esas pruebas, cuenta como acierto; si no la lanza, como fallo.
- `assert passed == total`: dentro del alcance declarado, todas deben pasar.

In [ ]:
SOF_REPO = EXT_DIR / "sql-on-fhir.js"
if not SOF_REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/FHIR/sql-on-fhir.js", str(SOF_REPO)], check=True)
sof_commit = subprocess.run(["git", "-C", str(SOF_REPO), "rev-parse", "HEAD"], check=True,
                            capture_output=True, text=True).stdout.strip()

IMPLEMENTED = ["basic", "collection", "combinations", "constant", "constant_types", "fhirpath", "fhirpath_numbers",
               "fn_empty", "fn_extension", "fn_first", "fn_join", "fn_oftype", "fn_reference_keys", "foreach",
               "logic", "union", "validate", "view_resource", "where"]


def norm(rows):
    return sorted(json.dumps(r, sort_keys=True) for r in rows)


results = []
for suite_name in IMPLEMENTED:
    suite = json.loads((SOF_REPO / "tests" / f"{suite_name}.json").read_text(encoding="utf-8"))
    for t in suite["tests"]:
        try:
            rows = run_view(t["view"], suite["resources"])
            ok = (not t.get("expectError")) and norm(rows) == norm(t["expect"])
        except Exception:
            ok = bool(t.get("expectError"))
        results.append({"suite": suite_name, "test": t["title"], "passed": ok})
conformance = pd.DataFrame(results)
conformance["commit"] = sof_commit
conformance.to_csv(REPORTS_DIR / "sof_conformance.csv", index=False)
passed, total = int(conformance["passed"].sum()), len(conformance)
print(f"sql-on-fhir.js @ {sof_commit[:10]}: {passed}/{total} pruebas oficiales dentro del alcance")
assert passed == total, conformance[~conformance["passed"]]
conformance.groupby("suite")["passed"].agg(["sum", "count"])

### Paso 6 — Escribir las `ViewDefinition`s

**Explicación línea por línea**

- `"resource": "Patient"`: cada vista describe **un solo** tipo de recurso.
- `{"name": "patient_id", "path": "getResourceKey()"}`: la clave primaria de la tabla.
- `"path": "subject.getReferenceKey(Patient)"`: la clave del paciente referenciado. Así la tabla de condiciones se une con la de pacientes **sin** tocar cadenas a mano.
- `"deceased.ofType(dateTime)"`: `deceased[x]` puede ser booleano o fecha; `ofType` toma solo la variante fecha. Igual con `onset.ofType(dateTime)` y `effective.ofType(dateTime)`.
- `"code.coding.where(system='http://snomed.info/sct').code.first()"`: el código SNOMED CT, elegido **por sistema** y no por posición. Si mañana el emisor añade un segundo código (doble codificación, que viste en el curso), la vista sigue siendo correcta. `first()` cumple la regla de un valor por columna.
- `observation_quantity`: `"where": [{"path": "value.ofType(Quantity).exists()"}]` deja solo observaciones con valor numérico; `value.ofType(Quantity).value` y `.code` dan valor y unidad UCUM.
- `observation_component`: `"forEach": "component"` crea una fila por componente (sistólica y diastólica de una presión arterial) y conserva el código del panel padre (`parent_code`).
- `"status": "active"`: elemento obligatorio de `ViewDefinition` (estado de publicación).
- `(VIEWS_DIR / f"{v['name']}.json").write_text(...)`: cada vista queda como archivo en `views/`. **Esto sí va al repo**: es la especificación reutilizable de tus tablas.

In [ ]:
SCT = "http://snomed.info/sct"
LN = "http://loinc.org"

VIEWS = [
    {"resourceType": "ViewDefinition", "name": "patient", "status": "active", "resource": "Patient",
     "select": [{"column": [
         {"name": "patient_id", "path": "getResourceKey()"},
         {"name": "gender", "path": "gender"},
         {"name": "birth_date", "path": "birthDate"},
         {"name": "deceased_ts", "path": "deceased.ofType(dateTime)"}]}]},
    {"resourceType": "ViewDefinition", "name": "encounter", "status": "active", "resource": "Encounter",
     "select": [{"column": [
         {"name": "encounter_id", "path": "getResourceKey()"},
         {"name": "patient_id", "path": "subject.getReferenceKey(Patient)"},
         {"name": "class_code", "path": "class.code"},
         {"name": "start_ts", "path": "period.start"},
         {"name": "end_ts", "path": "period.end"}]}]},
    {"resourceType": "ViewDefinition", "name": "condition", "status": "active", "resource": "Condition",
     "select": [{"column": [
         {"name": "condition_id", "path": "getResourceKey()"},
         {"name": "patient_id", "path": "subject.getReferenceKey(Patient)"},
         {"name": "encounter_id", "path": "encounter.getReferenceKey(Encounter)"},
         {"name": "snomed_code", "path": f"code.coding.where(system='{SCT}').code.first()"},
         {"name": "clinical_status", "path": "clinicalStatus.coding.code.first()"},
         {"name": "onset_ts", "path": "onset.ofType(dateTime)"},
         {"name": "abatement_ts", "path": "abatement.ofType(dateTime)"}]}]},
    {"resourceType": "ViewDefinition", "name": "observation_quantity", "status": "active", "resource": "Observation",
     "where": [{"path": "value.ofType(Quantity).exists()"}],
     "select": [{"column": [
         {"name": "observation_id", "path": "getResourceKey()"},
         {"name": "patient_id", "path": "subject.getReferenceKey(Patient)"},
         {"name": "encounter_id", "path": "encounter.getReferenceKey(Encounter)"},
         {"name": "effective_ts", "path": "effective.ofType(dateTime)"},
         {"name": "loinc_code", "path": f"code.coding.where(system='{LN}').code.first()"},
         {"name": "value_num", "path": "value.ofType(Quantity).value"},
         {"name": "unit", "path": "value.ofType(Quantity).code"}]}]},
    {"resourceType": "ViewDefinition", "name": "observation_component", "status": "active", "resource": "Observation",
     "select": [
         {"column": [
             {"name": "observation_id", "path": "getResourceKey()"},
             {"name": "patient_id", "path": "subject.getReferenceKey(Patient)"},
             {"name": "effective_ts", "path": "effective.ofType(dateTime)"},
             {"name": "parent_code", "path": f"code.coding.where(system='{LN}').code.first()"}]},
         {"forEach": "component", "column": [
             {"name": "loinc_code", "path": f"code.coding.where(system='{LN}').code.first()"},
             {"name": "value_num", "path": "value.ofType(Quantity).value"},
             {"name": "unit", "path": "value.ofType(Quantity).code"}]}]},
]
for v in VIEWS:
    (VIEWS_DIR / f"{v['name']}.json").write_text(json.dumps(v, indent=2), encoding="utf-8")
print("Vistas guardadas:", sorted(p.name for p in VIEWS_DIR.glob("*.json")))

### Paso 7 — Ejecutar las vistas sobre el NDJSON

**Explicación línea por línea**

- `def read_ndjson(path):` es un **generador** (`yield`): entrega un recurso por línea sin cargar el archivo entero. Así se procesan exportaciones de millones de recursos con memoria constante.
- `for v in VIEWS:` cada vista lee solo el archivo de su tipo (`v["resource"]`).
- `pd.DataFrame(rows, columns=_view_columns(v))`: fija el orden de columnas aunque una vista no produzca filas.
- `.to_parquet(SOF_DIR / f"{v['name']}.parquet", index=False)`: cada vista materializada como tabla.
- El tiempo por vista te da una idea del costo real de evaluar FHIRPath en Python; un runner dentro de la base de datos traduce las vistas a SQL y suele ser mucho más rápido.

In [ ]:
def read_ndjson(path):
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            if line.strip():
                yield json.loads(line)


def _view_columns(view):
    return [c for sel in view["select"] for c in _columns(sel)]


view_counts = {}
for v in VIEWS:
    t0 = time.time()
    rows = run_view(v, read_ndjson(NDJSON_DIR / f"{v['resource']}.ndjson"))
    df = pd.DataFrame(rows, columns=_view_columns(v))
    df.to_parquet(SOF_DIR / f"{v['name']}.parquet", index=False)
    view_counts[v["name"]] = len(df)
    print(f"{v['name']:<22} {len(df):>9,} filas  {time.time() - t0:6.1f} s")

## Parte C — Capa analítica: SQL sobre las vistas

### Paso 8 — Conciliar con las tablas de la semana 4

En la semana 4 aplanaste los mismos Bundles con funciones de Python escritas a mano. Si ambas rutas son correctas, deben dar exactamente lo mismo.

**Explicación línea por línea**

- `sof_raw.*`: las vistas tal como salieron del runner. Las fechas siguen siendo **texto ISO 8601 con desfase** (`1971-01-29T04:04:48-05:00`): la capa de vistas no debe perder información.
- `sof.*` con `SELECT * REPLACE (CAST(col AS TIMESTAMPTZ) AT TIME ZONE 'UTC' AS col)`: la capa analítica tipa las fechas y las lleva a UTC sin zona, la misma convención de las semanas 3 y 4. `REPLACE` sustituye solo esas columnas y deja el resto igual.
- `s4.*`: las tablas de `silver_fhir` de la semana 4.
- `checks = {...}`: cada consulta devuelve una cifra en las dos rutas:
  - pacientes, encuentros y condiciones: conteos;
  - `obs_quantity`: en la semana 4, las filas con valor numérico y sin `parent_code` son observaciones con `valueQuantity`;
  - `obs_component`: las filas con `parent_code` son componentes;
  - `sum_value`: la suma de todos los valores numéricos, redondeada; detecta un valor distinto aunque los conteos coincidan.
- `EXCEPT`: `SELECT ... FROM sof.condition EXCEPT SELECT ... FROM s4.condition` devuelve las filas que están en una tabla y no en la otra. Que dé 0 en ambos sentidos prueba igualdad fila a fila de esas columnas.
- `assert recon["sof"].equals(recon["semana4"])`: si no coinciden, el notebook se detiene.

In [ ]:
TS_COLS = {"patient": ["deceased_ts"], "encounter": ["start_ts", "end_ts"], "condition": ["onset_ts", "abatement_ts"],
           "observation_quantity": ["effective_ts"], "observation_component": ["effective_ts"]}
con = duckdb.connect()
con.execute("CREATE SCHEMA sof_raw; CREATE SCHEMA sof; CREATE SCHEMA s4")
for v in VIEWS:
    name = v["name"]
    con.execute(f"CREATE OR REPLACE VIEW sof_raw.{name} AS SELECT * FROM read_parquet('{(SOF_DIR / (name + '.parquet')).as_posix()}')")
    casts = ", ".join(f"CAST({c} AS TIMESTAMPTZ) AT TIME ZONE 'UTC' AS {c}" for c in TS_COLS[name])
    con.execute(f"CREATE OR REPLACE VIEW sof.{name} AS SELECT * REPLACE ({casts}) FROM sof_raw.{name}")
for t in ["patient", "encounter", "condition", "observation"]:
    con.execute(f"CREATE OR REPLACE VIEW s4.{t} AS SELECT * FROM read_parquet('{(DATA_DIR / 'silver_fhir' / (t + '.parquet')).as_posix()}')")

checks = {
    "patients": ("SELECT COUNT(*) FROM sof.patient", "SELECT COUNT(*) FROM s4.patient"),
    "encounters": ("SELECT COUNT(*) FROM sof.encounter", "SELECT COUNT(*) FROM s4.encounter"),
    "conditions": ("SELECT COUNT(*) FROM sof.condition", "SELECT COUNT(*) FROM s4.condition"),
    "obs_quantity": ("SELECT COUNT(*) FROM sof.observation_quantity",
                     "SELECT COUNT(*) FROM s4.observation WHERE parent_code IS NULL AND value_num IS NOT NULL"),
    "obs_component": ("SELECT COUNT(*) FROM sof.observation_component",
                      "SELECT COUNT(*) FROM s4.observation WHERE parent_code IS NOT NULL"),
    "sum_value": ("SELECT ROUND(SUM(value_num), 2) FROM (SELECT value_num FROM sof.observation_quantity "
                  "UNION ALL SELECT value_num FROM sof.observation_component)",
                  "SELECT ROUND(SUM(value_num), 2) FROM s4.observation"),
    "condition_rows_only_in_sof": (
        "SELECT COUNT(*) FROM (SELECT condition_id, patient_id, snomed_code, onset_ts FROM sof.condition "
        "EXCEPT SELECT condition_id, patient_id, snomed_code, onset_ts FROM s4.condition)", "SELECT 0"),
    "condition_rows_only_in_s4": (
        "SELECT COUNT(*) FROM (SELECT condition_id, patient_id, snomed_code, onset_ts FROM s4.condition "
        "EXCEPT SELECT condition_id, patient_id, snomed_code, onset_ts FROM sof.condition)", "SELECT 0"),
}
recon = pd.DataFrame([{"check": k, "sof": con.execute(a).fetchall()[0][0], "semana4": con.execute(b).fetchall()[0][0]}
                      for k, (a, b) in checks.items()])
print(recon.to_string(index=False))
assert recon["sof"].equals(recon["semana4"]), "Las dos rutas no coinciden"

### Paso 9 — Una pregunta analítica: última presión sistólica de cada paciente con hipertensión esencial

**Explicación línea por línea (SQL)**

- `WITH htn AS (SELECT DISTINCT patient_id FROM sof.condition WHERE snomed_code = '59621000')`: pacientes con hipertensión esencial (el mismo código semilla de la semana 6).
- `JOIN sof.observation_component c USING (patient_id)`: une por la clave que produjo `getReferenceKey(Patient)`. **Aquí está la división de trabajo de SQL on FHIR:** la vista no une; el motor SQL sí.
- `WHERE c.loinc_code = '8480-6'`: componente sistólico (LOINC) del panel de presión arterial.
- `QUALIFY ROW_NUMBER() OVER (PARTITION BY patient_id ORDER BY effective_ts DESC) = 1`: la última medición por paciente (semana 3). Se ordena por la fecha ya convertida a UTC en el esquema `sof`; ordenar el texto ISO 8601 sería incorrecto si dos mediciones tienen desfases horarios distintos.
- `CAST(effective_ts AS TIMESTAMP)`: iguala la precisión de las fechas (microsegundos). Según la versión de pandas con que escribiste la semana 4, el Parquet puede guardar nanosegundos; sin el `CAST`, los valores serían iguales pero los tipos no, y la comparación fallaría.
- La misma consulta sobre las tablas de la semana 4 debe dar el mismo resultado (`assert`).

In [ ]:
def last_sbp(schema, comp_table, comp_filter):
    return con.execute(f"""
    WITH htn AS (SELECT DISTINCT patient_id FROM {schema}.condition WHERE snomed_code = '59621000')
    SELECT patient_id, CAST(effective_ts AS TIMESTAMP) AS effective_ts, value_num AS sbp
    FROM {schema}.{comp_table} c JOIN htn USING (patient_id)
    WHERE c.loinc_code = '8480-6' {comp_filter}
    QUALIFY ROW_NUMBER() OVER (PARTITION BY patient_id ORDER BY effective_ts DESC) = 1
    ORDER BY patient_id
    """).df()


sbp_sof = last_sbp("sof", "observation_component", "")
sbp_s4 = last_sbp("s4", "observation", "AND c.parent_code IS NOT NULL")
assert sbp_sof.equals(sbp_s4)
print(f"{len(sbp_sof)} pacientes con hipertensión esencial y presión sistólica registrada")
sbp_sof.describe()

### Paso 10 — La alternativa rápida: SQL directo sobre el JSON, y sus trampas

DuckDB puede leer NDJSON y extraer campos con rutas JSON. Es útil para explorar, pero la ruta JSON es **posicional y literal**: no sabe de sistemas de códigos ni de tipos de elección.

**Explicación línea por línea**

- `read_ndjson_objects('...')`: lee cada línea como un valor JSON en la columna `json`, sin inferir un esquema.
- `json_extract_string(json, '$.code.coding[0].code')`: el primer código de la lista, **sea del sistema que sea**.
- La primera consulta compara los conteos por código con la vista: con Synthea coinciden porque cada condición trae un solo código.
- `dual = {...}`: una condición escrita a mano con **doble codificación** (el ejemplo del curso de la ADHA: ICD-10-AM `E11.2` y SNOMED CT-AU `127013003`), con el código ICD-10-AM primero. ICD-10-AM es una clasificación con licencia de IHACPA; aquí aparece un solo código como ilustración.
- `json_path_answer`: la ruta `coding[0]` devuelve `E11.2`, un código de **clasificación**, no el de SNOMED CT.
- `fhirpath_answer`: la vista, que elige por sistema, devuelve `127013003`.
- **Tipos de elección:** en JSON el valor está en `valueQuantity` o en `valueCodeableConcept` según el caso; una ruta JSON debe conocer la clave concreta. FHIRPath expresa la intención (`value.ofType(Quantity)`) y el modelo R4 resuelve la clave.

In [ ]:
ndjson_condition = (NDJSON_DIR / "Condition.ndjson").as_posix()
json_counts = con.execute(f"""
SELECT json_extract_string(json, '$.code.coding[0].code') AS code, COUNT(*) AS n
FROM read_ndjson_objects('{ndjson_condition}') GROUP BY 1 ORDER BY 1
""").df()
view_counts_by_code = con.execute("SELECT snomed_code AS code, COUNT(*) AS n FROM sof.condition GROUP BY 1 ORDER BY 1").df()
print("SQL sobre JSON == vista, en Synthea:", json_counts.equals(view_counts_by_code))

dual = {"resourceType": "Condition", "id": "dual-demo", "subject": {"reference": "Patient/demo"},
        "code": {"coding": [{"system": "http://hl7.org/fhir/sid/icd-10-am", "code": "E11.2"},
                            {"system": SCT, "code": "127013003"}]}}
dual_path = NDJSON_DIR / "dual_coding_demo.ndjson"
dual_path.write_text(json.dumps(dual) + "\n", encoding="utf-8")
json_path_answer = con.execute(
    f"SELECT json_extract_string(json, '$.code.coding[0].code') FROM read_ndjson_objects('{dual_path.as_posix()}')").fetchall()[0][0]
fhirpath_answer = run_view(VIEWS[2], [dual])[0]["snomed_code"]
print("Ruta JSON coding[0]:", json_path_answer, "| Vista FHIRPath por sistema:", fhirpath_answer)
assert json_path_answer == "E11.2" and fhirpath_answer == "127013003"

### Paso 11 — La misma pregunta con la API REST de FHIR

La API REST está pensada para leer y escribir recursos, normalmente de un paciente o de pocos registros. Para responder la pregunta del Paso 9 por API necesitarías:

1. `GET [base]/Patient?_has:Condition:patient:code=http://snomed.info/sct|59621000` para los pacientes (búsqueda con encadenamiento inverso `_has`), recorriendo todas las páginas;
2. por **cada** paciente, `GET [base]/Observation?patient=<id>&code=http://loinc.org|8480-6&_sort=-date&_count=1`.

**Explicación línea por línea**

- `n_requests = 1 + len(sbp_sof)`: como mínimo una petición por paciente más la búsqueda inicial (sin contar páginas). Con SQL fue **una** consulta sobre datos ya exportados.
- `LIVE`: si hay acceso al servidor público HAPI (semana 4), se prueba la búsqueda `_has` con `_summary=count`, que devuelve solo el total. Los datos de ese servidor **no** son los tuyos: el objetivo es ver que la sintaxis funciona, no comparar cifras.
- **Nunca** envíes datos reales a un servidor público de pruebas.

In [ ]:
import requests

n_requests = 1 + len(sbp_sof)
print(f"Por API REST: al menos {n_requests} peticiones HTTP; por SQL: 1 consulta")

FHIR_BASE = os.environ.get("FHIR_BASE", "https://hapi.fhir.org/baseR4")
try:
    r = requests.get(f"{FHIR_BASE}/Patient", params={"_has:Condition:patient:code": f"{SCT}|59621000",
                                                     "_summary": "count"},
                     headers={"Accept": "application/fhir+json"}, timeout=30)
    r.raise_for_status()
    print("Servidor público:", FHIR_BASE, "| pacientes con 59621000:", r.json().get("total"))
except Exception as exc:
    print(f"Servidor público no disponible ({type(exc).__name__}); la comparación de arriba no depende de él")

## Parte D — Cuándo sí y cuándo no (criterios objetivos)

**API REST de FHIR (búsqueda e interacciones)**

- **Sí:** leer o escribir datos de un paciente o de pocos recursos en el momento (una app clínica, una integración entre sistemas, validar o enviar un recurso). Es el uso para el que está diseñada.
- **No:** recorrer una población entera para analizarla. Necesitas una petición por página y por paciente (Paso 11), dependes de los parámetros de búsqueda que el servidor implemente y la búsqueda no agrega datos: `_summary=count` da un total, no un promedio.

**Exportación masiva (`$export`, NDJSON)**

- **Sí:** extraer una población o un grupo completo hacia un repositorio analítico. Es asíncrona y entrega archivos por tipo de recurso.
- **No:** consultas en tiempo real de un paciente; el trabajo se completa en segundo plano y los datos quedan "congelados" en el momento de la exportación.

**SQL on FHIR (`ViewDefinition`)**

- **Sí:** convertir FHIR en tablas de forma **portable y compartible**: la misma vista puede ejecutarse con otro runner u otro motor. Las vistas resuelven tipos de elección, sistemas de códigos y referencias con FHIRPath (Pasos 6 y 10).
- **No:** uniones, ordenamiento o agregaciones dentro de la vista. La especificación los deja al motor SQL. Tampoco sustituye un modelo común con vocabularios estándar cuando el objetivo es un estudio en red.

**SQL directo sobre el JSON**

- **Sí:** exploración rápida, una pregunta puntual.
- **No:** como base de un pipeline. Depende del dialecto de cada motor y se equivoca en silencio con listas y tipos de elección (Paso 10).

**OMOP CDM (semana 8)**

- **Sí:** investigación observacional reproducible y estudios multicéntricos con las herramientas de OHDSI, con vocabularios estándar y conceptos mapeados.
- **No:** intercambio clínico operativo. Exige un ETL completo y un mapeo de vocabularios con su propio control de calidad.

**Regla práctica:** API para operar sobre un paciente; `$export` para mover poblaciones; vistas SQL on FHIR para convertir FHIR en tablas; SQL para unir y agregar; OMOP para investigación con vocabularios estándar.

## Cierre

**Preguntas para tu bitácora**

1. ¿Por qué una `ViewDefinition` no permite uniones, y dónde se hacen?
2. ¿Qué pasa con la columna `snomed_code` si el emisor añade un segundo código y tú usaste `coding[0]`?
3. ¿Por qué el runner debe **fallar** cuando una columna no marcada como colección devuelve dos valores?
4. Tu runner no implementa `repeat`. Busca en la especificación para qué sirve y piensa un recurso de Synthea donde haría falta.
5. Para un tablero que muestra cada mañana los ingresos del día de un hospital, ¿qué combinación de API, `$export`, vistas y SQL elegirías y por qué?

**Commit sugerido:** `feat(s05): exportación NDJSON, view runner SQL on FHIR con pruebas oficiales y conciliación en DuckDB`

**Qué sube al repo:** el notebook y `views/*.json`. **Qué no:** nada de `data/`.

**Fuentes:** [SQL on FHIR (HL7)](https://build.fhir.org/ig/HL7/sql-on-fhir/en/index.html) · [ViewDefinition](https://build.fhir.org/ig/HL7/sql-on-fhir/en/StructureDefinition-ViewDefinition.html) · [Pruebas oficiales (sql-on-fhir.js)](https://github.com/FHIR/sql-on-fhir.js) · [Patrón asíncrono y NDJSON (FHIR R4)](https://hl7.org/fhir/R4/async.html) · [Búsqueda FHIR R4 (`_has`, `_summary`)](https://www.hl7.org/fhir/R4/search.html) · [fhirpathpy](https://pypi.org/project/fhirpathpy/)